# BlueDot Technical AI Safety Puzzle 1

This notebook is for investigating the hidden layer representation in the BlueDot Technical AI Safety puzzle.

The model predicts eight independent binary features from text. Seven features are linearly represented at the chosen layer. One feature is represented differently. The task is to identify that feature and explain the geometry.

## Feature labels

- `number`
- `question`
- `color`
- `food`
- `sentiment`
- `country`
- `person`
- `body_part`

## Architecture

The model uses `sentence-transformers/all-MiniLM-L6-v2`, mean pooling into a 384 dimensional vector, then an MLP head.

The MLP is `384 -> 64 -> 64 -> 64 -> 64 -> 8`. The final eight outputs are logits. Each logit is passed through a sigmoid independently.

The layer to inspect is hidden layer 2 after ReLU. This gives a 64 dimensional activation vector.

Architecture diagram source: https://raw.githubusercontent.com/SamDower/bluedot-tais-puzzle/main/model_architecture.png

## Setup

Run this cell first. It clones the upstream puzzle repo containing the trained model and data.

In [ ]:
!pip install -q sentence-transformers torch scikit-learn matplotlib numpy
![ -d /content/bluedot-tais-puzzle ] || git clone https://github.com/SamDower/bluedot-tais-puzzle.git /content/bluedot-tais-puzzle
%cd /content/bluedot-tais-puzzle

## Load model and get hidden layer activations

In [ ]:
import json
import torch
import torch.nn as nn
import numpy as np
from sentence_transformers import SentenceTransformer

class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(384, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 8),
        )

    def forward(self, x):
        return self.layers(x)

enc = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
m = Head()
m.load_state_dict(torch.load('model.pt', map_location='cpu', weights_only=False))
m.eval()
feature_names = json.load(open('feature_names.json'))
feature_names

In [ ]:
def load_jsonl(path):
    texts = []
    labels = []
    with open(path) as f:
        for line in f:
            item = json.loads(line)
            texts.append(item['text'])
            labels.append(item['labels'])
    return texts, np.array(labels, dtype=np.int64)

train_texts, y_train = load_jsonl('data/train.jsonl')
test_texts, y_test = load_jsonl('data/test.jsonl')

with torch.no_grad():
    X_train_emb = torch.from_numpy(enc.encode(train_texts, convert_to_numpy=True, show_progress_bar=True))
    X_test_emb = torch.from_numpy(enc.encode(test_texts, convert_to_numpy=True, show_progress_bar=True))
    X_train = m.layers[:6](X_train_emb).numpy()
    X_test = m.layers[:6](X_test_emb).numpy()

X_train.shape, X_test.shape, y_train.shape, y_test.shape

## Task 1

Train one linear probe per feature. The feature whose linear probe fails or behaves oddly is the candidate non-linear feature.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

results = []
for i, name in enumerate(feature_names):
    clf = LogisticRegression(max_iter=2000, class_weight='balanced')
    clf.fit(X_train, y_train[:, i])
    pred = clf.predict(X_test)
    prob = clf.predict_proba(X_test)[:, 1]
    results.append((name, accuracy_score(y_test[:, i], pred), roc_auc_score(y_test[:, i], prob)))

for row in sorted(results, key=lambda x: x[1]):
    print(f'{row[0]:12s} accuracy={row[1]:.4f} auc={row[2]:.4f}')

## Task 2

After identifying the odd feature, inspect the geometry. Useful checks include PCA, nonlinear classifiers, clustering by label, and checking whether two or more linear directions together recover the feature.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

# Replace this after looking at the probe results.
feature_to_inspect = 0

mlp = MLPClassifier(hidden_layer_sizes=(32,), max_iter=1000, random_state=0)
mlp.fit(X_train, y_train[:, feature_to_inspect])
print(classification_report(y_test[:, feature_to_inspect], mlp.predict(X_test)))

pca = PCA(n_components=2)
Z = pca.fit_transform(X_test)
print('PCA shape:', Z.shape)

## Task 3

Train a model with a stranger representation, then explain and defend why the representation is more interesting.